# GravitySort - GPU Sorting & ML Reduction Framework

**Kaggle GPU Notebook** | Builds and benchmarks CUDA C++ kernels on T4/P100.

## Section 0 - GPU & Environment Check

In [ ]:
import subprocess, os, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out)
    return r

print('='*60)
print('GPU INFO')
print('='*60)
run('nvidia-smi')
run('nvcc --version')
run('cmake --version')
run('ninja --version')
print('CPUs:', os.cpu_count())


## Section 1 - Download Project

In [ ]:
import os

ZIP_URL     = 'https://github.com/ThronAxis/GravitySort/archive/refs/heads/main.zip'
ZIP_PATH    = '/kaggle/working/GravitySort.zip'
PROJECT_DIR = '/kaggle/working/GravitySort-main'

if os.path.isdir(PROJECT_DIR):
    print('Already downloaded:', PROJECT_DIR)
else:
    print('Downloading GravitySort from GitHub...')
    r = run(f'wget -q --show-progress -O {ZIP_PATH} "{ZIP_URL}"')
    if r.returncode != 0:
        print('wget failed, trying curl...')
        r = run(f'curl -L -o {ZIP_PATH} "{ZIP_URL}"')
    assert r.returncode == 0, 'Download failed - check Internet is ON in Settings'
    print('Extracting...')
    run(f'unzip -q {ZIP_PATH} -d /kaggle/working/')

assert os.path.isdir(PROJECT_DIR), f'Not found: {PROJECT_DIR}'
print('Project files:')
run(f'find {PROJECT_DIR} -type f | sort')


## Section 1b - Install Dependencies

In [ ]:
import shutil, glob

print('='*55)
print('DEPENDENCY AUDIT')
print('='*55)

# System tools
need_apt = []
for t in ['nvcc','cmake','ninja','ncu','nsys','git']:
    p = shutil.which(t)
    print(f'  {"OK  " if p else "MISS"} {t:<8} {p or ""}')
    if not p and t in ('cmake','ninja'): need_apt.append(t)

# Install missing tools
if need_apt:
    print('Installing:', need_apt)
    run('apt-get update -qq')
    run('apt-get install -y -q cmake ninja-build')

run('apt-get install -y -q libglfw3-dev libglew-dev 2>/dev/null || true')

# Python packages
print('\nPython packages:')
for p in ['numpy','matplotlib','scipy','pygame']:
    try:
        m = __import__(p); print(f'  OK   {p} {m.__version__}')
    except ImportError:
        print(f'  Installing {p}...'); run(f'pip install {p} -q')

# CUDA libs
print('\nCUDA libraries:')
for lib in ['libcublas.so','libcudart.so']:
    found = glob.glob(f'/usr/local/cuda/**/{lib}*', recursive=True)
    print(f'  {"OK  " if found else "MISS"} {lib}')

thrust_ok = os.path.isdir('/usr/local/cuda/include/thrust')
print(f'  {"OK  " if thrust_ok else "MISS"} Thrust headers')
print('\nReady for build.')


## Section 2 - CMake Build (4-6 minutes)

In [ ]:
import subprocess, os

PROJECT_DIR = '/kaggle/working/GravitySort-main'
BUILD_DIR   = '/kaggle/working/build'
os.makedirs(BUILD_DIR, exist_ok=True)

# ── Diagnostics first ────────────────────────────────────────
print('== Pre-build checks ==')
print('PROJECT_DIR exists:', os.path.isdir(PROJECT_DIR))
print('CMakeLists.txt exists:', os.path.isfile(f'{PROJECT_DIR}/CMakeLists.txt'))
r = subprocess.run('nvcc --version', shell=True, capture_output=True, text=True)
print('nvcc:', r.stdout.strip().split('\n')[-1] if r.returncode==0 else 'NOT FOUND')
r = subprocess.run('cmake --version', shell=True, capture_output=True, text=True)
print('cmake:', r.stdout.strip().split('\n')[0] if r.returncode==0 else 'NOT FOUND')
r = subprocess.run('ninja --version', shell=True, capture_output=True, text=True)
print('ninja:', r.stdout.strip() if r.returncode==0 else 'NOT FOUND - installing...')
if r.returncode != 0:
    subprocess.run('apt-get install -y -q ninja-build', shell=True)

# ── GPU arch ─────────────────────────────────────────────────
smi = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f'\nGPU: {smi}')
arch_map = {'T4':'75','V100':'70','P100':'60','A100':'80','A6000':'86','RTX 30':'86','RTX 40':'89'}
arch = next((v for k,v in arch_map.items() if k.lower() in smi.lower()), '75')
print(f'CUDA arch: sm_{arch}')

# ── CMake configure ───────────────────────────────────────────
print('\n== CMake Configure ==')
cmd = (
    f'cmake -S {PROJECT_DIR} -B {BUILD_DIR}'
    f' -DCMAKE_CUDA_ARCHITECTURES={arch}'
    f' -DCMAKE_BUILD_TYPE=Release'
    f' -G Ninja'
)
print('CMD:', cmd)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout[-3000:] if r.stdout else '')
if r.stderr: print('STDERR:', r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError(f'CMake configure failed (exit {r.returncode})')

# ── Build ─────────────────────────────────────────────────────
ncpu = os.cpu_count() or 4
print(f'\n== CMake Build (parallel={ncpu}) ==')
r = subprocess.run(f'cmake --build {BUILD_DIR} --parallel {ncpu}',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-4000:] if r.stdout else '')
if r.stderr: print('STDERR:', r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError(f'Build failed (exit {r.returncode})')

print('\nBuild complete! Executables:')
r = subprocess.run(f'ls -lh {BUILD_DIR}/', shell=True, capture_output=True, text=True)
print(r.stdout)


## Section 3 - Sorting Kernels

In [ ]:
BUILD_DIR = '/kaggle/working/build'

for algo, exe in [('BITONIC','bitonic_sort'),('RADIX','radix_sort'),('ODD-EVEN','odd_even_sort')]:
    print(f'\n{"="*50}')
    print(f'{algo} SORT')
    print('='*50)
    sizes = [1<<20, 1<<24, 1<<26] if algo != 'ODD-EVEN' else [65536]
    for N in sizes:
        print(f'N={N:,}')
        run(f'{BUILD_DIR}/{exe} {N}')


## Section 4 - ML Reduction Kernels

In [ ]:
BUILD_DIR = '/kaggle/working/build'
print('='*50)
print('REDUCTION: 4 variants vs thrust::reduce')
print('='*50)
for N in [1<<24, 1<<25, 1<<27]:
    print(f'\nN={N:,}')
    run(f'{BUILD_DIR}/reduction_demo {N}')


## Section 5 - Memory Optimization Demos

In [ ]:
BUILD_DIR = '/kaggle/working/build'
print('-- Bank Conflict Demo --')
run(f'{BUILD_DIR}/shared_mem_demo')
print('\n-- Stream Concurrency Demo --')
run(f'{BUILD_DIR}/streams_demo')


## Section 6 - GravityTensor Operations

In [ ]:
BUILD_DIR = '/kaggle/working/build'
run(f'{BUILD_DIR}/tensor_demo')


## Section 7 - Google Benchmark

In [ ]:
BUILD_DIR = '/kaggle/working/build'
print('-- Sort Benchmark --')
run(f'{BUILD_DIR}/bench_sort --benchmark_format=console --benchmark_repetitions=3')
print('\n-- Reduce Benchmark --')
run(f'{BUILD_DIR}/bench_reduce --benchmark_format=console --benchmark_repetitions=3')


## Section 8 - Nsight Profiling

In [ ]:
BUILD_DIR = '/kaggle/working/build'
NCU = 'ncu --set full --clock-control none --target-processes all'
print('-- Profiling: bitonic_sort --')
run(f'{NCU} {BUILD_DIR}/bitonic_sort 4194304')
print('\n-- Nsight Systems: streams --')
run(f'nsys profile --trace=cuda,nvtx --output=/kaggle/working/streams_report {BUILD_DIR}/streams_demo')
print('Profile saved: /kaggle/working/streams_report.nsys-rep')


## Section 9 - Roofline Model

In [ ]:
import sys
PROJECT_DIR = '/kaggle/working/GravitySort-main'
sys.path.insert(0, f'{PROJECT_DIR}/profiling')
run(f'python {PROJECT_DIR}/profiling/roofline.py --output /kaggle/working/roofline.png')
from IPython.display import Image, display
display(Image('/kaggle/working/roofline.png'))


## Section 10 - Sort Animation

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

N = 64
arr = np.random.permutation(N).astype(float)

def bitonic_steps(a):
    steps = [a.copy()]; n = len(a); k = 2
    while k <= n:
        j = k // 2
        while j >= 1:
            for i in range(n):
                l = i ^ j
                if l > i:
                    asc = (i & k) == 0
                    if (asc and a[i] > a[l]) or (not asc and a[i] < a[l]):
                        a[i], a[l] = a[l], a[i]
            steps.append(a.copy()); j //= 2
        k *= 2
    return steps

steps = bitonic_steps(arr.copy())
steps = steps[::max(1, len(steps)//60)]

matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
for spine in ax.spines.values(): spine.set_color('#30363d')
ax.tick_params(colors='#8b949e')
ax.set_title(f'GravitySort Bitonic Sort  N={N}', color='#f0f6fc', fontsize=13, fontweight='bold')
colors = plt.cm.plasma(np.linspace(0.1, 0.95, N))
bars = ax.bar(range(N), steps[0], color=colors, width=0.85, edgecolor='none')
ax.set_ylim(0, N+2)
info = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='#8b949e', fontsize=9, va='top')

def update(frame):
    data = steps[frame]
    sc = plt.cm.plasma(data / N)
    for bar, h, c in zip(bars, data, sc):
        bar.set_height(h); bar.set_color(c)
    info.set_text(f'Step {frame+1}/{len(steps)}')
    return bars

ani = animation.FuncAnimation(fig, update, frames=len(steps), interval=80, blit=False)
plt.tight_layout()
HTML(ani.to_jshtml())


## Section 11 - Unit Tests

In [ ]:
BUILD_DIR = '/kaggle/working/build'
print('='*50)
print('UNIT TESTS')
print('='*50)
r1 = run(f'{BUILD_DIR}/test_sort   --gtest_color=yes')
r2 = run(f'{BUILD_DIR}/test_reduce --gtest_color=yes')
print()
if r1.returncode == 0 and r2.returncode == 0:
    print('ALL TESTS PASSED')
else:
    print('SOME TESTS FAILED - check output above')
